In [ ]:
!pip install wandb torchvision thop

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import matplotlib.pyplot as plt
import wandb
from thop import profile

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


#Login to Weights & Biases

In [ ]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: m25csa002 (m25csa002-iit-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

#Initialize W&B run

In [ ]:
wandb.init(
    project="cifar10-gradient-flow",
    name="resnet18-run",
    config={
        "dataset": "CIFAR-10",
        "model": "ResNet18",
        "epochs": 40,
        "batch_size": 128,
        "learning_rate": 0.001,
        "optimizer": "Adam",
        "seed": 42
    }
)

#Custom CIFAR-10 Dataset

In [ ]:
class CustomCIFAR10(Dataset):
    def __init__(self, train=True, transform=None):
        self.transform = transform
        self.dataset = torchvision.datasets.CIFAR10(
            root="./data",
            train=train,
            download=True
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

#DataLoaders

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

base_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True
)

num_samples = len(base_dataset)
indices = np.random.permutation(num_samples)

split = int(0.8 * num_samples)

train_idx = indices[:split]
val_idx   = indices[split:]

train_dataset = Subset(
    CustomCIFAR10(train=True, transform=train_transform),
    train_idx
)

val_dataset = Subset(
    CustomCIFAR10(train=True, transform=eval_transform),
    val_idx
)

test_dataset = CustomCIFAR10(
    train=False,
    transform=eval_transform
)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:03<00:00, 43.7MB/s]


Train size: 40000
Val size: 10000
Test size: 10000


#Model (ResNet-18 adapted for CIFAR-10)

In [ ]:
model = resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, 10)

model = model.to(device)

#FLOPs & Parameter Count

In [ ]:
dummy_input = torch.randn(1, 3, 32, 32).to(device)
flops, params = profile(model, inputs=(dummy_input,), verbose=False)

print(f"FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"Params: {params/1e6:.2f} Million")

wandb.log({
    "FLOPs (MFLOPs)": flops / 1e6,
    "Parameters (Millions)": params / 1e6
})

FLOPs: 557.89 MFLOPs
Params: 11.17 Million


#Loss & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#Gradient Flow Visualization

In [ ]:
def plot_gradient_flow(named_parameters):
    ave_grads = []
    layers = []

    for n, p in named_parameters:
        if p.requires_grad and p.grad is not None:
            layers.append(n)
            ave_grads.append(
                p.grad.abs().mean().detach().cpu().numpy()
            )

    fig = plt.figure(figsize=(10,5))
    plt.plot(ave_grads)
    plt.xticks(range(len(layers)), layers, rotation="vertical")
    plt.xlabel("Layers")
    plt.ylabel("Average Gradient")
    plt.title("Gradient Flow")
    plt.tight_layout()
    return fig

#Weight Update Flow Visualization

In [ ]:
def plot_weight_update_flow(model, old_params):
    updates = []
    layers = []

    for (name, param), old_param in zip(model.named_parameters(), old_params):
        layers.append(name)
        updates.append(
            (param - old_param).abs().mean().detach().cpu().numpy()
        )

    fig = plt.figure(figsize=(10,5))
    plt.plot(updates)
    plt.xticks(range(len(layers)), layers, rotation="vertical")
    plt.xlabel("Layers")
    plt.ylabel("Mean Weight Update")
    plt.title("Weight Update Flow")
    plt.tight_layout()
    return fig

#Evaluation Function

In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            loss_sum += loss.item()
            _, pred = out.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return loss_sum / len(loader), 100. * correct / total


#Training (30 Epochs)

In [ ]:
epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    old_params = [p.clone().detach() for p in model.parameters()]

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, pred = out.max(1)
        total += y.size(0)
        correct += pred.eq(y).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total
    val_loss, val_acc = evaluate(model, val_loader)
    test_loss, test_acc = evaluate(model, test_loader)

    grad_fig = plot_gradient_flow(model.named_parameters())
    weight_fig = plot_weight_update_flow(model, old_params)

    wandb.log({
        "Epoch": epoch + 1,
        "Train Loss": train_loss,
        "Train Accuracy": train_acc,
        "Validation Loss": val_loss,
        "Validation Accuracy": val_acc,
        "Test Accuracy": test_acc,
        "Gradient Flow": wandb.Image(grad_fig),
        "Weight Update Flow": wandb.Image(weight_fig)
    })

    plt.close("all")

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Val Acc: {val_acc:.2f}% | "
          f"Test Acc: {test_acc:.2f}%")

wandb.finish()

Epoch [1/30] | Train Acc: 47.60% | Val Acc: 55.40% | Test Acc: 54.61%
Epoch [2/30] | Train Acc: 65.53% | Val Acc: 71.04% | Test Acc: 70.80%
Epoch [3/30] | Train Acc: 73.39% | Val Acc: 74.64% | Test Acc: 75.02%
Epoch [4/30] | Train Acc: 77.86% | Val Acc: 76.75% | Test Acc: 76.67%
Epoch [5/30] | Train Acc: 80.64% | Val Acc: 78.48% | Test Acc: 78.59%
Epoch [6/30] | Train Acc: 82.62% | Val Acc: 80.72% | Test Acc: 80.97%
Epoch [7/30] | Train Acc: 84.50% | Val Acc: 83.05% | Test Acc: 82.80%
Epoch [8/30] | Train Acc: 85.70% | Val Acc: 83.96% | Test Acc: 83.73%
Epoch [9/30] | Train Acc: 87.00% | Val Acc: 85.04% | Test Acc: 84.41%
Epoch [10/30] | Train Acc: 88.11% | Val Acc: 84.90% | Test Acc: 84.50%
Epoch [11/30] | Train Acc: 88.83% | Val Acc: 85.15% | Test Acc: 84.99%
Epoch [12/30] | Train Acc: 89.66% | Val Acc: 85.29% | Test Acc: 84.48%
Epoch [13/30] | Train Acc: 90.55% | Val Acc: 87.16% | Test Acc: 86.71%
Epoch [14/30] | Train Acc: 90.72% | Val Acc: 87.38% | Test Acc: 87.12%
Epoch [15/30] |

Epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
FLOPs (MFLOPs),▁
Parameters (Millions),▁
Test Accuracy,▁▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇██▇▇██████████
Train Accuracy,▁▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█████████████
Train Loss,█▆▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇██▇▇██████████
Validation Loss,█▄▄▃▃▃▂▂▂▂▂▂▁▁▁▂▁▁▂▂▁▁▁▁▁▁▁▁▂▁
Epoch,30
FLOPs (MFLOPs),557.88902
Parameters (Millions),11.17396
